[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ralbu85/cbm-colab-2026/blob/main/notebooks/3-1_%EC%97%B4%ED%99%94%EB%8D%B0%EC%9D%B4%ED%84%B0%EC%99%80_%EA%B1%B4%EC%A0%84%EC%84%B1%EC%A7%80%ED%91%9C.ipynb)

# 3-1 · 압출기 필터 데이터의 구조 — 실험·1초 기록·10초 표, 그리고 건전성 지표

**⏱ 90분** · 모듈 3: 열화 지표 → ③ 잔여수명(RUL) 예측 → 교체 시점 결정

## 🎯 이번 시간의 질문

1. 이 데이터는 **무엇을, 어떻게** 기록했고, 파일마다 **한 행은 무엇**인가? (앞 35분)
2. 필터가 막혀 가는 정도를 **숫자 하나(건전성 지표)**로 나타내고, 열화를 **언제 감지**할 수 있을까? (뒤 55분)

모듈 2는 "지금 어떤 상태인가"를 판단했습니다. 정비를 **계획**하려면 "앞으로 얼마나 버틸까", 즉 **잔여수명(RUL, Remaining Useful Life)**이 필요합니다. 그러려면 설비가 **새것일 때부터 고장 날 때까지** 기록한 데이터가 있어야 합니다.
2-1에서처럼 먼저 실험을 이해하고, 구조 그림을 본 뒤, **파일을 하나씩 열어** 한 행이 무엇인지 확인합니다. 그다음 막혀 가는 모습을 **건전성 지표(HI, Health Indicator)**로 만들고, 열화가 처음 감지되는 시점(P)부터 고장(F)까지의 시간을 봅니다.

| 이번 시간 | 코드 |
|---|---|
| ✍️ **직접 입력 (8개)** | 교차표 · 파일 열어 보기 `shape`·`head` (2번) · 실험별 행 수 `groupby().size()` · 한 필터 그래프 · 이동평균 `rolling` · `groupby().last()` · 조건을 만족하는 첫 시점 |
| ▶ **제공** — 실행하고 **결과만 읽는** 코드 | 파일 읽기, 1초 기록과 10초 표 비교, 다른 센서 그림, 41개 필터 겹쳐 그리기, P–F 함수 |

## 0. 장치와 실험 (읽기, 5분)

호주 멜버른대 연구팀이 2026년에 공개한 **압출기 필터 고장 실험**입니다(00에서 연습용으로 본 실험 목록 60회).

```
                    모터 (전압 voltage_v · 전류 current_a) ── 스크루를 돌림
                      │
원료 투입 ──▶ [ 가열 통 속의 스크루 :  뒤 back_c ─ 가운데 middle_c ─ 앞 front_c ] ──▶ 필터(금속망) ──▶ 다이 ──▶ 필라멘트
(feed_c)                                                                          ↑ 가루가 쌓여 막힘
```

연구자들은 이렇게 실험했습니다.

1. **압출기**: 플라스틱 알갱이를 넣으면 가열된 통 안에서 스크루가 돌며 녹여 앞으로 밀어냅니다. 녹은 플라스틱은 **필터(스테인리스 금속망)**를 지나 가는 실(지름 1.75mm 필라멘트)로 나옵니다. 필터는 이물질을 걸러 제품 불량을 막는 소모품입니다.
2. **일부러 오염시키기**: 원료에 **탄산칼슘 가루**를 섞어 가루가 필터에 쌓여 빨리 막히게 했습니다. 현장에서 오래 걸릴 막힘을 수 분~수십 분으로 **앞당긴 가속 실험**입니다.
3. **실험 1회 = 필터 1개의 일생**: 새 필터를 끼우고 **고장 날 때까지** 운전하며 **1초마다** 모터 전압·전류와 네 곳의 온도를 기록했습니다. 온도는 연구자가 정한 설정값으로 제어했습니다.
4. **고장 기록**: 실험마다 **고장 시각 F**(초)와 고장 형태를 적었습니다. 이것이 정답입니다.
   - **막힘(clog)**: 필터가 꽉 막혀 필라멘트가 더 나오지 않음 (처리량 0)
   - **파열(burst)**: 금속망이 찢어져 이물질이 제품으로 들어감

## 1. 데이터 구조 한눈에 보기 (읽기, 3분)

```
연구자가 기록한 것                       이 수업의 파일                             한 행                  크기
───────────────────────                  ───────────────────────────────────        ───────────────────    ───────────────
실험마다 고장 형태·고장 시각 F   ────▶  ① filter_runs.csv        실험 목록       실험 1회 (필터 1개)    60행 ×  5열
실험마다 1초 간격 센서 기록      ──┬─▶  ② filter_seconds.csv.gz  1초 기록        실험 1회의 1초         27,778행 ×  8열
                                   │
                                   └─▶  ③ filter_windows.csv     10초 표         실험 1회의 10초        2,746행 × 13열
                                         (②를 10초씩 평균 + ①의 고장 시각·고장 형태·분석 구분을 붙임)
```

- 세 파일은 **실험 번호 `run_id`**로 이어집니다. ①은 실험 하나가 한 행이고, ②·③은 실험 하나가 **여러 행**(시간 순서)입니다.
- 이번 모듈의 분석은 주로 **③ 10초 표**로 합니다.

이 순서대로 하나씩 열어 봅니다: **① 실험 목록 → ② 1초 기록 → ③ 10초 표 → 분석 대상 41회**.

In [ ]:
# ▶ 준비 셀 — 그대로 실행하세요. (실습 데이터와 한글 글꼴을 내려받습니다)
import os, urllib.request, warnings
REPO = "https://raw.githubusercontent.com/ralbu85/cbm-colab-2026/main/"
for path in ["data/filter_runs.csv",
             "data/filter_seconds.csv.gz",
             "data/filter_windows.csv",
             "fonts/NanumGothic.ttf"]:
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        urllib.request.urlretrieve(REPO + path, path)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_manager.fontManager.addfont("fonts/NanumGothic.ttf")
plt.rcParams.update({"font.family": "NanumGothic", "axes.unicode_minus": False,
                     "figure.figsize": (8, 4), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.max_columns", 20)
warnings.filterwarnings("ignore")
print("준비 완료!")

In [ ]:
# ▶ 그대로 실행하세요
runs = pd.read_csv('data/filter_runs.csv')              # ① 실험 목록
seconds = pd.read_csv('data/filter_seconds.csv.gz')     # ② 1초 기록
windows = pd.read_csv('data/filter_windows.csv')        # ③ 10초 표
print('읽기 완료')

## 2. ① 실험 목록 — 실험 1회 = 필터 1개

00에서 `runs.head()`로 본 표입니다(60행 × 5열).

| 열 | 뜻 |
|---|---|
| `run_id` | 실험 번호 (필터 한 개 = 실험 한 번) |
| `failure_mode` | 고장 형태: `clog` 막힘 / `burst` 파열 |
| `failure_s` | 고장 시각 F(초) = 이 필터의 수명 |
| `status` | 분석 사용 여부: `main` 분석 대상 / 그 밖은 제외 |
| `split` | 학습용(train) / 평가용(test) / 제외(excluded) |

어떤 실험을 분석하는지 고장 형태 × 분석 사용 여부로 교차해 셉니다(1-2에서 쓴 `pd.crosstab`).

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
print(runs.shape)
pd.crosstab(runs['failure_mode'], runs['status'])
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 **결과 해설** — 막힘 42회 중 **41회**(`main`)를 분석합니다.
- **파열 18회**(`burst_extension`)는 뺍니다. 망이 갑자기 찢어지는 다른 고장이라, 막힘이 진행되는 모습을 배우는 데 섞이면 안 됩니다.
- 막힘 1회(`time_mismatch`, 29번)는 뺍니다. 센서 기록은 812초에서 끝나는데 고장 시각은 866초로 적혀 있어, 정답(F)과 기록이 맞지 않습니다.

분석 대상 41회는 **학습용 33회 / 평가용 8회**로 나눠 두었습니다(`split`). 필터 단위로 나눴으므로 한 필터의 기록은 한쪽에만 들어 있습니다.

## 3. ② 1초 기록 — 센서 원자료

**P1 · 읽고 보기**

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
print(seconds.shape)
seconds.head()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 **결과 해설** — 27,778행 × 8열입니다. **한 행 = 실험 1회의 1초**입니다. 60회 실험 모두 0초부터 고장 시각까지 1초마다 한 행씩 있습니다(24번 필터는 F = 425초라서 0~425초, 426행).

| 열 | 뜻 |
|---|---|
| `run_id` | 실험 번호 (① 실험 목록과 같은 번호) |
| `elapsed_s` | 실험 시작 후 경과 시간(초) |
| `voltage_v`, `current_a` | 모터 전압(V), 모터 전류(A) |
| `feed_c`, `back_c`, `middle_c`, `front_c` | 원료 투입구·통 뒤·가운데·앞의 온도(°C) |

온도 네 열이 35 · 160 · 180 · 190°C 근처인 것은 연구자가 정한 **설정값**이기 때문입니다(6절에서 다시 봅니다).

## 4. ③ 10초 표 — 1초 기록을 10초씩 평균한 것

**P1 · 읽고 보기**

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
print(windows.shape)
windows.head()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 **결과 해설** — 2,746행 × 13열입니다. **한 행 = 실험 1회의 10초**입니다. 앞의 다섯 행은 1번 실험의 10초, 20초, …입니다.

| 열 | 뜻 |
|---|---|
| `run_id`, `time_s` | 실험 번호, 경과 시간(초). 10초마다 한 행 |
| `voltage_v` ~ `front_c` | ② 1초 기록의 **직전 10초 평균** (`time_s` = 10인 행 = 0~9초의 평균) |
| `failure_s`, `failure_mode`, `status`, `split` | ① 실험 목록에서 붙인 정보 (같은 실험의 모든 행에 같은 값) |
| `RUL_s` | 실제 잔여수명 = `failure_s` − `time_s` (3-2에서 사용) |

**②와 ③이 정말 이어지는지** 24번 필터로 확인합니다(▶). 1초 기록의 0~9초 평균과 10초 표의 첫 행을 비교하고, 두 기록을 겹쳐 그립니다.

In [ ]:
# ▶ 그대로 실행하세요
s24 = seconds[seconds['run_id'] == 24]
w24 = windows[windows['run_id'] == 24]
print('② 1초 기록 0~9초 전류 평균 :', round(s24[s24['elapsed_s'] < 10]['current_a'].mean(), 3))
print('③ 10초 표 time_s = 10 행   :', w24['current_a'].iloc[0])

plt.figure(figsize=(9, 4))
plt.plot(s24['elapsed_s'], s24['current_a'], color='lightgray', label='② 1초 기록')
plt.plot(w24['time_s'], w24['current_a'], 'o-', markersize=3, label='③ 10초 평균 (한 행)')
plt.xlabel('경과 시간 (초)')
plt.ylabel('모터 전류 (A)')
plt.title('24번 실험 · 1초 기록을 10초씩 평균해 한 행으로')
plt.legend()
plt.show()

💬 **결과 해설** — 두 값이 0.628 A로 같습니다. 1초 기록(회색)은 잡음으로 크게 오르내리고, 10초 평균(파랑)은 흐름이 더 잘 보입니다. 10초 행은 **그 시각까지 이미 기록된 값**만 평균하므로 현장에서 실시간으로 계산할 수 있는 값입니다.

## 5. 분석 대상 41회 — 실험마다 행이 몇 개인가

10초 표에서 분석 대상(`main`)만 골라 `main`으로 둡니다. 1-1에서 디스크마다 관찰된 날 수가 달랐듯이, 필터마다 행 수도 다릅니다. **P3 · 실험별 요약**으로 행 수와 수명을 나란히 봅니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
main = windows[windows['status'] == 'main'].copy()      # 분석 대상 41회
n_rows = main.groupby('run_id').size()
life = main.groupby('run_id')['failure_s'].first()
table = pd.DataFrame({'행 수': n_rows, '수명 F(초)': life})
print('필터', len(table), '개 · 전체', len(main), '행')
print(table.head())
table.describe().round()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


✅ **체크포인트** — 필터 41개 · 전체 2,190행, 수명 최소 115 · 중앙값 473 · 최대 1,596초.

💬 **결과 해설** — **행 수 = 수명 ÷ 10**(소수점 버림)입니다. 3번 필터는 905초를 버텨 90행입니다.
- 수명이 **115초에서 1,596초까지 10배 넘게** 다릅니다. 같은 조건으로 시험했는데도 그렇습니다(00의 "고정 주기 교체의 한계").
- 그래서 **오래 버틴 필터일수록 행이 많습니다.** 가장 긴 3개 필터가 전체 행의 17%입니다. 2,190행은 필터 2,190개가 아니라 **41개**입니다(1-1의 "행 수 ≠ 설비 수"). 3-3의 머신러닝에서 이 점이 다시 중요해집니다.

## 6. 필터 한 개의 일생

**P2 → P4** — 학습용 **24번 필터**(F = 425초)만 골라 전류 변화를 그립니다. `plt.axvline`으로 고장 시점에 세로선을 긋습니다.
(`.copy()`는 골라낸 표를 따로 복사해 두는 것입니다. 뒤에서 새 열을 추가할 때 필요합니다.)

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
one = main[main['run_id'] == 24].copy()
F = one['failure_s'].iloc[0]
plt.plot(one['time_s'], one['current_a'], marker='.')
plt.axvline(F, color='red', linestyle='--', label=f'고장 F = {F}초')
plt.xlabel('경과 시간 (초)')
plt.ylabel('모터 전류 (A)')
plt.legend()
plt.show()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 **결과 해설** — 처음 150초 정도는 0.6 A 근처에서 평평하다가, 이후 전류가 꾸준히 올라 고장 직전 약 1.15 A가 됩니다. 필터에 가루가 쌓일수록 녹은 플라스틱을 밀어내기 힘들어져 **모터가 더 많은 전류를 씁니다.** 열화 과정이 전류에 그대로 드러납니다.

다른 센서도 같이 변할까요? 같은 필터의 전압과 통 앞쪽 온도를 봅니다(▶).

In [ ]:
# ▶ 그대로 실행하세요
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].plot(one['time_s'], one['voltage_v'], marker='.')
axes[0].set_ylabel('모터 전압 (V)')
axes[1].plot(one['time_s'], one['front_c'], marker='.')
axes[1].set_ylabel('통 앞쪽 온도 (°C)')
for ax in axes:
    ax.axvline(F, color='red', linestyle='--')
    ax.set_xlabel('경과 시간 (초)')
fig.tight_layout()
plt.show()

💬 **결과 해설** — 전압은 약 9.6 V → 11.3 V로 **전류처럼 오릅니다.** 막힌 필터를 밀어내느라 모터 부하가 커지기 때문입니다. 통 앞쪽 온도는 190°C를 중심으로 약 90초 주기로 189~192°C를 오르내릴 뿐, 막힘과 함께 변하지 않습니다. 온도 제어기가 히터를 켜고 끄며 **설정값을 유지**하는 모습입니다.
데이터가 만들어진 방법을 알면, 어떤 열은 **열화 신호**(전류·전압)이고 어떤 열은 **운전 조건**(온도)인지 구분할 수 있습니다.

## 7. 건전성 지표(HI) 만들기 — 잡음 줄이기

10초 평균 전류도 아직 오르내림이 있습니다. **이동평균**으로 최근 3개(30초) 값을 평균하면 잡음이 더 줄어듭니다. 이렇게 **열화 정도를 보여 주는 숫자 하나**가 건전성 지표(HI)입니다.
`rolling(3, min_periods=1).mean()` = "최근 3개씩 평균, 처음처럼 값이 3개가 안 되면 있는 만큼만 평균".

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
one['hi'] = one['current_a'].rolling(3, min_periods=1).mean()
plt.plot(one['time_s'], one['current_a'], alpha=0.5, label='10초 평균 전류')
plt.plot(one['time_s'], one['hi'], linewidth=2, label='HI (최근 30초 이동평균)')
plt.xlabel('경과 시간 (초)')
plt.ylabel('전류 (A)')
plt.legend()
plt.show()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 HI(주황)는 흔들림이 줄어 **추세가 더 잘 보입니다.** 대신 과거 값을 평균하므로 변화를 조금 **늦게** 따라갑니다. 잡음 제거와 반응 속도는 교환 관계입니다(과제 2).

### ▶ 41회 실험 전체에 HI 만들기

같은 계산을 모든 실험에 적용합니다. `groupby('run_id')`로 **실험마다 따로** 이동평균을 계산해야 앞 실험의 값이 다음 실험에 섞이지 않습니다(5절: 한 실험이 여러 행). 학습용 33개 필터를 **고장 시점에 맞춰** 겹쳐 그립니다.

In [ ]:
# ▶ 그대로 실행하세요
main['hi'] = main.groupby('run_id')['current_a'].transform(lambda s: s.rolling(3, min_periods=1).mean())
train = main[main['split'] == 'train']
test = main[main['split'] == 'test']

plt.figure(figsize=(9, 4))
for rid, run in train.groupby('run_id'):
    plt.plot(run['time_s'] - run['failure_s'], run['hi'], alpha=0.5)
plt.xlabel('고장까지 남은 시간의 반대 (t − F, 초) · 0 = 고장')
plt.ylabel('HI (A)')
plt.title('학습용 33개 필터의 HI — 고장 시점(0)에 맞춰 정렬')
plt.xlim(-800, 0)
plt.show()

💬 **결과 해설** — 모든 필터가 고장(0)을 향해 HI가 올라가는 **공통 경향**이 있습니다. 하지만
- 처음 전류(0.57~0.78 A)와 오르기 시작하는 시점·속도가 제각각이고,
- 어떤 필터는 일찍 올라간 뒤 **오랫동안 평평하게** 버티다 고장 나며,
- 고장 순간의 HI도 제각각입니다.

"HI가 몇 A가 되면 고장"이라는 **고장 기준선이 하나로 딱 정해지지 않는다**는 뜻입니다. 얼마나 흩어져 있는지 숫자로 확인합니다.

## 8. 고장 기준선 — 고장 순간의 HI는 얼마인가

**P3** — 학습용 실험마다 마지막 행(고장 직전)의 HI를 `groupby(...).last()`로 뽑습니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
end_hi = train.groupby('run_id')['hi'].last()
print(end_hi.describe().round(3))
plt.hist(end_hi, bins=10)
plt.xlabel('고장 직전 HI (A)')
plt.ylabel('실험 수')
plt.show()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


✅ **체크포인트** — 중앙값(50%) 1.069 A, 최소 0.835 A, 최대 1.177 A.

💬 **결과 해설** — 절반의 필터는 1.07 A보다 낮은 HI에서 고장 났습니다. 기준선을 1.07 A로 잡으면 **절반은 기준선에 닿기 전에 고장** 납니다. 기준선을 낮게(보수적으로) 잡을수록 안전하지만 일찍 교체하게 됩니다.
다음 시간(3-2)에는 이 기준선(중앙값 1.07 A)을 이용해 "HI가 언제 기준선에 닿을까"로 RUL을 추정합니다.

## 9. 열화 감지 시점 P와 P–F 구간

1-2에서 배운 **P–F 구간**을 여기서도 계산합니다. 규칙: **"HI가 처음 30초 평균보다 0.1 A 이상 올라가면 열화 감지(P)"**.
7절에서 처음 전류가 필터마다 달랐으므로, 공통 기준값 대신 **필터 자신의 초기 상태**를 정상 기준으로 삼습니다. 2-2의 이상탐지와 같은 원리입니다.

**조건을 만족하는 첫 시점 찾기** — 조건으로 행을 고른 뒤(`P2`) 가장 이른 시간(`min()`)을 구합니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
base = one['current_a'].iloc[:3].mean()          # 처음 3개 행(30초)의 평균 = 이 필터의 정상 기준
P = one.loc[one['hi'] > base + 0.1, 'time_s'].min()
print('초기 기준:', round(base, 3), 'A')
print('감지 시점 P:', P, '초 / 고장 F:', F, '초 / P–F:', F - P, '초')
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 24번 필터는 **240초**에 열화가 감지되었고, 고장까지 **185초**의 여유가 있었습니다. 이 185초 안에 교체를 준비해야 합니다.

### ▶ 학습용 33개 필터 전체의 P–F 구간

같은 규칙을 모든 학습용 필터에 적용하는 함수입니다(`margin` = 초기 기준보다 얼마나 올라가면 감지할지).

In [ ]:
# ▶ 그대로 실행하세요
def detect_P(df, margin=0.1):
    """필터마다 열화 감지 시점 P, 고장 F, P–F 구간을 표로 만든다. 감지 못 하면 P는 빈칸(NaN)."""
    rows = []
    for rid, run in df.groupby('run_id'):
        base = run['current_a'].iloc[:3].mean()
        P = run.loc[run['hi'] > base + margin, 'time_s'].min()
        F = run['failure_s'].iloc[0]
        rows.append({'run_id': rid, 'P': P, 'F': F, 'PF': F - P})
    return pd.DataFrame(rows)

pf = detect_P(train, margin=0.1)
print('감지한 필터:', pf['P'].notna().sum(), '/', len(pf))
print('P–F 중앙값:', pf['PF'].median(), '초 / 최소:', pf['PF'].min(), '초')
plt.hist(pf['PF'], bins=12)
plt.xlabel('P–F 구간 (초)')
plt.ylabel('필터 수')
plt.show()

💬 **결과 해설** — 33개 모두 고장 전에 열화를 감지했고, P–F 구간의 중앙값은 **310초**입니다. 하지만 가장 짧은 필터는 **29초**뿐이었습니다(57번: 110초에 감지, 139초에 고장).
**P–F 구간도 필터마다 크게 다릅니다.** "감지되면 교체"라는 규칙만으로는 준비 시간이 부족한 필터가 생깁니다 → "감지 후 **얼마나 남았는지**"를 추정하는 RUL 예측이 필요한 이유입니다.

---
## 10. 실습 과제 (30분)

### 📝 과제 1 · 감지 기준(margin)을 바꾸면?
`detect_P(train, margin=...)`의 margin을 **0.05, 0.1, 0.2, 0.3**으로 바꿔 표를 채우세요.

| margin (A) | 감지한 필터 수 | P–F 중앙값(초) | P–F 최소(초) |
|---|---|---|---|
| 0.05 | | | |
| 0.1 | 33 | 310 | 29 |
| 0.2 | | | |
| 0.3 | | | |

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: for m in [0.05, 0.1, 0.2, 0.3]: pf_m = detect_P(train, margin=m) 후 위와 같이 출력
# (본문의 pf는 margin 0.1 결과로 과제 3에서 다시 쓰므로 덮어쓰지 않습니다.)


✏️ **나의 답:** margin을 키우면 어떻게 되나요? 너무 작으면 어떤 문제가 생길까요?

### 📝 과제 2 · 이동평균 창 크기의 영향
24번 필터에서 이동평균 창을 **1(평균 안 함), 3, 6**으로 바꿔 한 그림에 겹쳐 그리세요. 창을 키우면 무엇을 얻고 무엇을 잃나요?

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: for k in [1, 3, 6]: plt.plot(one['time_s'], one['current_a'].rolling(k, min_periods=1).mean(), label=f'창 {k}')


✏️ **나의 답:**

### 📝 과제 3 · 준비 시간이 120초 필요하다면?
필터 교체 준비(작업자 호출, 예비 필터 준비)에 **120초**가 걸린다고 합시다. margin = 0.1일 때, 학습용 33개 중 **P–F가 120초 이상인** 필터는 몇 개인가요? 나머지 필터는 어떻게 될까요?

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: (pf['PF'] >= 120).sum()


✏️ **나의 답:**

### 📝 과제 4 · 생각해 보기 — 가속 실험의 전류를 HI로 쓸 때
모터 전류를 필터의 건전성 지표로 쓸 때의 **장점 한 가지**와, 이 데이터가 **가루를 섞어 막힘을 앞당긴 실험**이라는 점을 생각할 때 **주의할 점 한 가지**를 적으세요.

✏️ **나의 답:**

---
## ✅ 정리

| 배운 것 | 핵심 |
|---|---|
| 실험 | 가루를 섞어 막힘을 앞당긴 실험. 필터마다 고장까지 운전(실험 1회 = 필터 1개의 일생), 1초마다 전압·전류·온도 기록 |
| 데이터 구조 | ① 실험 목록 60 × 5 (한 행 = 실험 1회) · ② 1초 기록 27,778 × 8 (한 행 = 실험의 1초) · ③ 10초 표 2,746 × 13 (한 행 = 실험의 10초, ②의 10초 평균 + ①의 정보). `run_id`로 이어짐 |
| 분석 대상 | 막힘 41회 (파열은 다른 고장, 기록과 정답이 어긋난 1회 제외). 행 수 = 수명 ÷ 10 → 긴 필터가 행이 많다 |
| 열화 신호 | 전류·전압은 막힘을 따라 오르고, 온도는 설정값으로 제어된 운전 조건 |
| 건전성 지표 HI | 전류의 30초 이동평균. 잡음↓ 대신 반응 지연↑ |
| 고장 기준선 | 고장 순간의 HI가 필터마다 달라(0.84~1.18 A) 하나의 값으로 정하기 어렵다 |
| P–F 구간 | 감지 후 고장까지. 중앙값 310초지만 최소 29초 → **감지만으로는 부족, 남은 시간 추정 필요** |

다음: **3-2 · RUL 예측 ① — 추세를 연장해서 고장선에 닿는 시간 구하기**